## Скрипт выгрузки данных Росстата по промпроизводству в обработке из отдельных помесячных файлов

In [1]:
# Выгрузим библиотеки
import pandas as pd
import requests
import os
from io import BytesIO
from datetime import datetime
from dateutil.relativedelta import relativedelta
from dotenv import load_dotenv
load_dotenv('data.env')

True

In [2]:
# Загрузим наш шаблон
df_monitor = pd.read_excel(os.getenv('MASK_FILE'))
df_monitor = df_monitor.iloc[:, 0:3]
df_monitor = df_monitor.dropna(how='all')
df_monitor.columns = df_monitor.iloc[0]
df_monitor = df_monitor.iloc[1:].reset_index(drop=True)
df_monitor = df_monitor.dropna(subset=['Код ОКПД 2'])
df_monitor['Код ОКПД 2'] = df_monitor['Код ОКПД 2'].astype(str)
df_monitor['v_okved'] = df_monitor['Код ОКПД 2'].str[0:2]
display(df_monitor.head(5))

1,Код ОКПД 2,Товар,ед.изм.,v_okved
0,26.11.3,Схемы интегральные электронные,Тысяча штук,26
1,26.2,Компьютеры и периферийное оборудование,Тысяча рублей,26
2,28.30.2,Тракторы для сельского хозяйства прочие,Штука,28
3,28.30.59.111,Комбайны зерноуборочные,Штука,28
4,28.41.11.001.АГ,Станки металлорежущие,Штука,28


In [3]:
# Функция выгрузки значения по заданному коду в заданном датафрейме
def value_okpd(okpd, df):
    try:
        temp_index = df.index[df['Unnamed: 1']==str(okpd)].tolist()
        if temp_index:  # Проверяем, что список не пустой
            temp_value = df.iloc[temp_index[0] + 2]['Unnamed: 2']
            return temp_value
        else:
            print(f"Код {okpd} не найден в файле")
            if len(okpd)==7:
                print('Проверяем на 7 знаков')
                okpd = okpd + '0'
                temp_index = df.index[df['Unnamed: 1']==okpd].tolist()
                if temp_index:
                    temp_value = df.iloc[temp_index[0] + 2]['Unnamed: 2']
                    return temp_value
                else:
                    print(f"Код {okpd} (с дополненным 0) не найден в файле")
    except IndexError:
        print('Тест ошибки с индексом')
    except Exception as e:
        print(f"Ошибка при обработке данных: {type(e).__name__}: {e}")
    temp_value = None
    return temp_value

In [4]:
# Функция выгрузки данных в соответствии с заданной структурой (месяцы-классы)
def fetch_data(months_dict):
    df_dict = {}
    # Список шаблонов имён файлов (порядок важен — от наиболее вероятного)
    url_patterns = [
        "http://ssl.rosstat.gov.ru/storage/mediabank/prom_{month}.xlsx",
        "http://ssl.rosstat.gov.ru/storage/mediabank/Prom_{month}.xlsx",
        "http://ssl.rosstat.gov.ru/storage/mediabank/prod_oper_{month_dash}.xlsx",
    ]

    for month in months_dict:
        try:
            print(f"Загрузка данных за период {month}")

            # month имеет формат mm_yyyy, для третьего шаблона нужен mm-yyyy
            month_dash = month.replace("_", "-")

            file_response = None
            used_url = None

            # Перебираем возможные варианты URL
            for pattern in url_patterns:
                url = pattern.format(month=month, month_dash=month_dash)
                print(f"  Проверка: {url}")
                try:
                    file_handshake = requests.head(url, verify=False, timeout=30)
                except requests.exceptions.RequestException as e:
                    print(f"  ✗ Ошибка при проверке {url}: {e}")
                    continue

                if file_handshake.status_code == 200:
                    print(f"  ✓ Найден файл: {url}")
                    file_response = requests.get(url, verify=False, timeout=30)
                    used_url = url
                    break
                else:
                    print(f"  Статус {file_handshake.status_code} для {url}")

            # Если ни один URL не подошёл — пропускаем месяц
            if file_response is None:
                print(f"Ошибка получения данных за {month}. Данные могут отсутствовать.")
                global month_not_exist
                month_not_exist = month
                print(f"Последний месяц: {month}")
                break

            file_excel = BytesIO(file_response.content)

            for code in months_dict.get(month):
                print(f"Загрузка данных по классу: {code}")
                sheet_combo = f"Н-П_{code}"
                df_dict[f"{month}_{code}"] = pd.read_excel(file_excel, sheet_name=sheet_combo)

        except requests.exceptions.Timeout:
            print(f"✗ Таймаут при обработке {month}")
        except requests.exceptions.ConnectionError:
            print(f"✗ Ошибка подключения")
        except requests.exceptions.RequestException as e:
            print(f"Произошла ошибка при выполнении запроса: {e}")
        except Exception as e:
            print(f"✗ Ошибка: {type(e).__name__}: {e}")
    return df_dict


In [5]:
# Функция создания структуры месяц-класс в соответствии с шаблоном и периодом загрузки
def get_month_dict(periods, mask_df):
    current_date = datetime.now()
    some_time_ago = current_date - relativedelta(months=periods)
    start_date = some_time_ago.replace(day=1)  # Первое число месяца
    
    dict_data={}
    temp_value=None
    
    # Генерируем даты по месяцам до текущего месяца
    while start_date <= current_date.replace(day=1):  # Только до текущего месяца
        month_str = start_date.strftime("%m_%Y")
        dict_data[month_str] = pd.Series(mask_df['v_okved'].dropna().unique()).sort_values().tolist()
        # Переходим к следующему месяцу
        start_date = start_date + relativedelta(months=1)
    return dict_data

In [6]:
# Создаем структуру дата-класс и загружаем данные
find_periods = int(os.getenv('PERIODS'))
months_practice=get_month_dict(find_periods, df_monitor)
global_df_dict = fetch_data(months_practice)
df_temp=pd.DataFrame(None)

Загрузка данных за период 05_2026
  Проверка: http://ssl.rosstat.gov.ru/storage/mediabank/prom_05_2026.xlsx
  ✓ Найден файл: http://ssl.rosstat.gov.ru/storage/mediabank/prom_05_2026.xlsx
Загрузка данных по классу: 10
Загрузка данных по классу: 16
Загрузка данных по классу: 17
Загрузка данных по классу: 18
Загрузка данных по классу: 20
Загрузка данных по классу: 22
Загрузка данных по классу: 24
Загрузка данных по классу: 26
Загрузка данных по классу: 28
Загрузка данных по классу: 29
Загрузка данных по классу: 30
Загрузка данных за период 06_2026
  Проверка: http://ssl.rosstat.gov.ru/storage/mediabank/prom_06_2026.xlsx
  ✓ Найден файл: http://ssl.rosstat.gov.ru/storage/mediabank/prom_06_2026.xlsx
Загрузка данных по классу: 10
Загрузка данных по классу: 16
Загрузка данных по классу: 17
Загрузка данных по классу: 18
Загрузка данных по классу: 20
Загрузка данных по классу: 22
Загрузка данных по классу: 24
Загрузка данных по классу: 26
Загрузка данных по классу: 28
Загрузка данных по классу:

In [7]:
# Наполняем целевой датафрейм
list_temp=[]
target_df = df_monitor.copy()
for key in months_practice: 
    if key == month_not_exist:
        break
    for code in df_monitor['Код ОКПД 2']:
        list_temp.append(value_okpd(code, global_df_dict[f"{key}_{str(code)[0:2]}"]))
    target_df[key] = list_temp
    list_temp=[]

# Грузим датафрейм в нужный файл
target_df.to_excel(os.getenv('OUTPUT_FILE'), sheet_name='Данные', index=False)

Код 26.11.3 не найден в файле
Проверяем на 7 знаков
Код 30.30.34.110 не найден в файле
Код 16.10.1 не найден в файле
Проверяем на 7 знаков
Код 18 не найден в файле
Код 30.20.20.140 + 30.20.32.130 не найден в файле
Код 26.11.3 не найден в файле
Проверяем на 7 знаков
Код 30.30.34.110 не найден в файле
Код 16.10.1 не найден в файле
Проверяем на 7 знаков
Код 18 не найден в файле
Код 30.20.20.140 + 30.20.32.130 не найден в файле
Код 26.11.3 не найден в файле
Проверяем на 7 знаков
Код 30.30.34.110 не найден в файле
Код 16.10.1 не найден в файле
Проверяем на 7 знаков
Код 18 не найден в файле
Код 30.20.20.140 + 30.20.32.130 не найден в файле
Код 26.11.3 не найден в файле
Проверяем на 7 знаков
Код 30.30.34.110 не найден в файле
Код 16.10.1 не найден в файле
Проверяем на 7 знаков
Код 18 не найден в файле
Код 30.20.20.140 + 30.20.32.130 не найден в файле
